# V'DJer — пилоты на подготовленных BAM

Ноутбук запускает V'DJer на подвыборке симулированных ридов и оценивает результат по эталону. Для каждого вида:

1. **Подготовка BAM** (`scripts/make_vdjer_bam.py`): подвыборка пар (seed 42) → STAR → сортированный индексированный BAM с невыровненными ридами. Готовый BAM не пересоздаётся. Состав BAM и обработка ридов описаны в `docs/vdjer_bam.md`.
2. **Пилот** (`scripts/run_vdjer_pilot.py`): V'DJer на BAM в двух режимах фильтра покрытия — `rf=1` (контиги, прошедшие проверку покрытия) и `rf=0` (непроверенные пути графа, только диагностика). Цепь пилота: IGL у человека, IGH у мыши. Режим с готовым `vdj_contigs.fa` не пересчитывается.
3. **Оценка** (`scripts/score_vdjer_pilot.py`): для шаблонов, чьи риды попали в подвыборку, ищется точное вхождение полной V…J и junction в контигах; результат — `score.json` в каталоге пилота.

Каталог пилота: `vdjer_pilots_20261006/<вид>_<образец>_<пары>/`. Эталон и FASTQ берутся из базовых веток симуляции (человек — `umicons_in5ng`, мышь — `ec1x3_in5ng`).

## Параметры

In [ ]:
import json, os, subprocess, sys
from pathlib import Path

ROOT = Path(os.environ.get("BCR_VOLUME", "/data/user/epishkin"))
SCRIPTS = next(p for p in (ROOT / "scripts", *(q / "scripts" for q in (Path.cwd(), *Path.cwd().parents)))
               if (p / "run_vdjer_pilot.py").exists())
sys.path.insert(0, str(SCRIPTS))
from make_vdjer_bam import DEFAULT_OUT as PILOTS

SPECIES = os.environ["BCR_SPECIES"].split(",") if os.environ.get("BCR_SPECIES") else ["human", "mouse"]
PAIRS = int(os.environ.get("BCR_VDJER_PAIRS", 300_000))
THREADS = int(os.environ.get("BCR_THREADS", 8))
RF = [1, 0]

def run_script(name, *args):
    cmd = [sys.executable, str(SCRIPTS / name), *map(str, args)]
    print("RUN:", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True, cwd=SCRIPTS)

print("скрипты:", SCRIPTS, "\nпилоты:", PILOTS, "\nвиды:", SPECIES, "пары:", PAIRS, "потоки:", THREADS)

## BAM для пилота

In [ ]:
for species in SPECIES:
    run_script("make_vdjer_bam.py", "--species", species, "--pairs", PAIRS, "--threads", THREADS)

## Пилот V'DJer

In [ ]:
for species in SPECIES:
    rf_args = [a for rf in RF for a in ("--rf", rf)]
    run_script("run_vdjer_pilot.py", "--species", species, "--pairs", PAIRS, "--threads", THREADS, *rf_args)

## Оценка по эталону

In [ ]:
for species in SPECIES:
    rf_args = [a for rf in RF for a in ("--rf", rf)]
    run_script("score_vdjer_pilot.py", "--species", species, "--pairs", PAIRS, *rf_args)

## Сводка

In [ ]:
import pandas as pd

rows = []
for score in sorted(PILOTS.glob(f"*_{PAIRS}/score.json")):
    report = json.loads(score.read_text())
    for mode, m in report["modes"].items():
        rows.append({"пилот": score.parent.name, "режим": mode, "пар": report["sampled_pairs"],
                     "проверенных V…J в подвыборке": report["represented_verified_unique_vj"],
                     "контигов": m["contigs"], "точных полных V…J": m["exact_full_unique_vj"],
                     "junction в контигах": m["exact_junctions_in_sequence"],
                     "junction в эталоне": report["represented_verified_unique_junctions"]})
pd.DataFrame(rows)